# 🚀 ShardFlow — Pipeline Node (Colab / GPU Node Setup)

Run this notebook on Google Colab (T4 / A100 GPU) to join a ShardFlow distributed cluster.

In [ ]:
# Step 1: Install ShardFlow dependencies
!pip install -q torch transformers tokenizers safetensors accelerate fastapi uvicorn requests pydantic sse-starlette

In [ ]:
# Step 2: Clone repository & install ShardFlow package
!git clone https://github.com/adityaraut/Shardflow.git /content/Shardflow 2>/dev/null || (cd /content/Shardflow && git pull)
%cd /content/Shardflow
!pip install -e .

In [ ]:
# Step 3: Run Node with Cloudflare Tunnel & Register with Topology Registry
import os, sys, time, requests, torch
from shardflow.transport.tunnel import start_cloudflare_tcp_tunnel
from shardflow.node.layer_loader import load_layer_slice
from shardflow.node.node import PipelineNode
import asyncio

# CONFIGURATION
MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0" # Or meta-llama/Meta-Llama-3-8B
REGISTRY_URL = "http://your-registry-url:8001"   # Railway / Render FastAPI Registry URL
NODE_ID = f"colab-node-{int(time.time())}"
LAYER_START = 0
LAYER_END = 11
IS_LAST_NODE = False
NEXT_NODE_HOST = None # Add next node host if intermediate
NEXT_NODE_PORT = None
LOCAL_PORT = 9000

print(f"[INFO] Starting Node {NODE_ID} for layers [{LAYER_START}, {LAYER_END})...")
vram_total = torch.cuda.get_device_properties(0).total_memory / 1e6 if torch.cuda.is_available() else 0
vram_free = torch.cuda.mem_get_info()[0] / 1e6 if torch.cuda.is_available() else 0

# Load model slice
model_slice = load_layer_slice(
    model_path=MODEL_ID,
    layer_start=LAYER_START,
    layer_end=LAYER_END,
    include_norm=IS_LAST_NODE,
    include_lm_head=IS_LAST_NODE,
    device="cuda" if torch.cuda.is_available() else "cpu",
)

node = PipelineNode(
    model_slice=model_slice,
    is_first_node=(LAYER_START == 0),
    is_last_node=IS_LAST_NODE,
    next_node_host=NEXT_NODE_HOST,
    next_node_port=NEXT_NODE_PORT,
    listen_host="0.0.0.0",
    listen_port=LOCAL_PORT,
)

# Start node
await node.serve_forever()